# Laufzeit- und Speichervergleich der beiden Aufbereitungsketten

Gemessen wird die Aufbereitung **einer** Nut je Kette. Die beiden Eingaben sind nicht
dieselbe Aufzeichnung und auch nicht gleich lang, deshalb ist die reine Laufzeit je
Durchlauf nur bedingt vergleichbar. Massgeblich sind die normierten Groessen am Ende:
Zeit je Million Ausgabezeilen und Spitzenspeicher je Ausgabezeile.

Was das Notebook misst:

| Groesse | Ermittlung |
|---|---|
| Laufzeit | `perf_counter`, Median aus mehreren Wiederholungen nach einem Warmlauf |
| Spitzenspeicher | RSS-Abtastung in einem Nebenthread, gegen die Grundlast vor dem Lauf |
| Ausgabezeilen | Zeilenzahl der erzeugten Tabelle |
| Eingabegroesse | Summe der gelesenen Rohdateien |

Der Spitzenspeicher wird ueber RSS gemessen und nicht ueber `tracemalloc`, weil beide
Ketten den groessten Teil ihres Speichers in C-Erweiterungen belegen (NumPy, Arrow,
DuckDB), die `tracemalloc` nicht sieht.

In [1]:
from __future__ import annotations

import gc
import os
import platform
import shutil
import sys
import threading
import time
from pathlib import Path

import pandas as pd

try:
    import psutil
except ImportError:
    raise SystemExit("psutil fehlt:  pip install psutil")

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 1  Konfiguration

Pfade und Wiederholungszahl. `REPEATS` bestimmt, wie oft jede Kette nach dem Warmlauf
ausgefuehrt wird; berichtet wird der Median.

In [ ]:
REPEATS = 5
WARMUP = 1

# Gegebene Raw-Recording für das alte Script
OLD_JSON = Path("../data/raw_old/sld_test_20250405-111231649.json")

# Gegebene Test-Recording des neuen Scripts
NEW_SESSION = Path("./new_data/SLD-Test_2026-07-17T13-38-45_591745")
NEW_RECORDING = NEW_SESSION / "recording_2026-07-17T13-38-55_822881"
NEW_METADATA = NEW_SESSION / "metadata.json"

# Zielverzeichnis der neuen Kette. Muss zu dem passen, was new_script schreibt.
NEW_OUTPUT_DIR = Path("./bench_out")

NUT, PLATTE = 1, 40

for p in (OLD_JSON, NEW_RECORDING, NEW_METADATA):
    print(f"{'vorhanden' if p.exists() else 'FEHLT    '}  {p}")

vorhanden  old_data\sld_test_20250405-111231649.json
vorhanden  new_data\SLD-Test_2026-07-17T13-38-45_591745\recording_2026-07-17T13-38-55_822881
vorhanden  new_data\SLD-Test_2026-07-17T13-38-45_591745\metadata.json


## 2  Adapter

Die beiden Funktionen kapseln je einen vollstaendigen Aufbereitungslauf und geben die
Zeilenzahl des Ergebnisses zurueck. Hier ist der einzige Ort, an dem die Schnittstellen
der beiden Skripte auftauchen; falls sie abweichen, nur diese beiden Zellen anpassen.

**Zum Umfang der beiden Laeufe.** Die alte Kette liefert die fertige Tabelle im
Arbeitsspeicher, die neue schreibt sie zusaetzlich als Datei. Damit beide dasselbe
leisten, wird der Schreibvorgang der neuen Kette in Abschnitt 5 gesondert ausgewiesen
und kann von der Gesamtzeit abgezogen werden.

In [ ]:
import sys
sys.path.insert(0, ".")

from pathlib import Path

SCRIPTS_DIR = Path(__file__).resolve().parent / ".." / "scripts"
sys.path.insert(0, str(SCRIPTS_DIR.resolve()))

import old_script

HF_META_XLSX = Path("../data/added_data/HFmeta_insight_hub.xlsx")
LF_META_CSV  = Path("../data/added_data/LF_variables_readable.csv")

for p in (HF_META_XLSX, LF_META_CSV):
    print(f"{'vorhanden' if p.exists() else 'FEHLT    '}  {p}")


def run_old() -> int:
    """Alte Kette: JSON einer Nut"""
    df, _meta = old_script.tidy_data(
        str(OLD_JSON),
        dateipfad_InsightHub_meta=str(HF_META_XLSX),
        dateipfad_lfncvar_meta=str(LF_META_CSV),
    )
    n = len(df)
    del df
    return n

vorhanden  HFmeta_insight_hub.xlsx
vorhanden  LF_variables_readable.csv


In [ ]:
import polars as pl
import sys
sys.path.insert(0, ".")

from pathlib import Path

BASE = Path(__file__).resolve().parent
sys.path.insert(0, str((BASE / ".." / "scripts").resolve()))

import cleaner


def run_new() -> int:
    """Neue Kette: Verzeichnis einer Nut -> fertige Tabelle. Gibt die Zeilenzahl zurueck.

    `overwrite=True` ist noetig, weil die Kette einen vorhandenen Ausgabestand sonst
    ueberspringt und der Lauf dann nichts messen wuerde.
    """
    out_path, _ = cleaner.cleaner(
        str(NEW_RECORDING), nut=NUT, platte=PLATTE, overwrite=True
    )
    n = pl.scan_parquet(out_path).select(pl.len()).collect().item()
    return n

## 3  Messvorrichtung

Ein Warmlauf fuellt den Dateisystem-Cache, damit die spaetere Messung nicht den ersten
Plattenzugriff mitzaehlt. Vor jedem Lauf wird die Garbage Collection angestossen, damit
die Grundlast reproduzierbar ist.

In [ ]:
def _peak_rss(fn, interval: float = 0.005):
    """
    Rueckgabe: (Ergebnis, Laufzeit in s, Spitzen-RSS ueber der Grundlast in MB).
    """
    proc = psutil.Process()
    gc.collect()
    baseline = proc.memory_info().rss
    peak = baseline
    stop = threading.Event()

    def sample():
        nonlocal peak
        while not stop.is_set():
            try:
                rss = proc.memory_info().rss
            except psutil.Error:
                break
            if rss > peak:
                peak = rss
            time.sleep(interval)

    watcher = threading.Thread(target=sample, daemon=True)
    watcher.start()
    t0 = time.perf_counter()
    try:
        result = fn()
    finally:
        elapsed = time.perf_counter() - t0
        stop.set()
        watcher.join()

    return result, elapsed, (peak - baseline) / 1e6


def measure(fn, label: str, repeats: int = REPEATS, warmup: int = WARMUP) -> dict:
    for _ in range(warmup):
        fn()

    times, peaks, rows = [], [], None
    for i in range(repeats):
        n, dt, mb = _peak_rss(fn)
        times.append(dt)
        peaks.append(mb)
        rows = n
        print(f"  {label}  Lauf {i + 1}/{repeats}: {dt:6.2f} s, {mb:7.1f} MB")

    times.sort()
    peaks.sort()
    return {
        "Kette": label,
        "Zeit_median_s": times[len(times) // 2],
        "Zeit_min_s": times[0],
        "Zeit_max_s": times[-1],
        "Speicher_median_MB": peaks[len(peaks) // 2],
        "Ausgabezeilen": rows,
    }


def dir_size_mb(*paths) -> float:
    total = 0
    for p in paths:
        p = Path(p)
        if p.is_file():
            total += p.stat().st_size
        elif p.is_dir():
            total += sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1e6

## 4  Messung

In [8]:
print("System:", platform.platform())
print("Python:", sys.version.split()[0])
print("Kerne :", psutil.cpu_count(logical=True), "logisch,",
      psutil.cpu_count(logical=False), "physisch")
print("RAM   :", f"{psutil.virtual_memory().total / 1e9:.1f} GB")
print()

results = []
print("Alte Kette")
results.append(measure(run_old, "alt (Pandas, JSON)"))
print("\nNeue Kette")
results.append(measure(run_new, "neu (DuckDB/Polars, Parquet)"))

df_res = pd.DataFrame(results)
df_res["Eingabe_MB"] = [
    dir_size_mb(OLD_JSON),
    dir_size_mb(NEW_RECORDING) + dir_size_mb(NEW_METADATA),
]
df_res

System: Windows-11-10.0.26200-SP0
Python: 3.12.10
Kerne : 16 logisch, 8 physisch
RAM   : 17.1 GB

Alte Kette
  alt (Pandas, JSON)  Lauf 1/5:   4.14 s,   979.3 MB
  alt (Pandas, JSON)  Lauf 2/5:   4.18 s,   975.4 MB
  alt (Pandas, JSON)  Lauf 3/5:   4.13 s,   973.1 MB
  alt (Pandas, JSON)  Lauf 4/5:   4.22 s,   966.8 MB
  alt (Pandas, JSON)  Lauf 5/5:   4.16 s,   965.1 MB

Neue Kette
175277 175273
WCS_Y_mm: 273 von 779,055 Zeilen ohne Zuordnung (Toleranz 10ms zu klein?)
175277 175273
WCS_Y_mm: 273 von 779,055 Zeilen ohne Zuordnung (Toleranz 10ms zu klein?)
  neu (DuckDB/Polars, Parquet)  Lauf 1/5:   0.56 s,    77.2 MB
175277 175273
WCS_Y_mm: 273 von 779,055 Zeilen ohne Zuordnung (Toleranz 10ms zu klein?)
  neu (DuckDB/Polars, Parquet)  Lauf 2/5:   0.55 s,    65.6 MB
175277 175273
WCS_Y_mm: 273 von 779,055 Zeilen ohne Zuordnung (Toleranz 10ms zu klein?)
  neu (DuckDB/Polars, Parquet)  Lauf 3/5:   0.55 s,   119.9 MB
175277 175273
WCS_Y_mm: 273 von 779,055 Zeilen ohne Zuordnung (Toleranz 1

,Kette,Zeit_median_s,Zeit_min_s,Zeit_max_s,Speicher_median_MB,Ausgabezeilen,Eingabe_MB
0,"alt (Pandas, JSON)",4.162,4.129,4.221,973.074,813359,6.469
1,"neu (DuckDB/Polars, Parquet)",0.555,0.546,0.569,96.387,779055,3.929


## 5  Schreibvorgang der neuen Kette

Die neue Kette schreibt ihr Ergebnis zusaetzlich als Datei, die alte gibt es nur im
Arbeitsspeicher zurueck. Der folgende Lauf misst den Schreibvorgang gesondert, damit
die Gesamtzeiten um diesen Anteil bereinigt verglichen werden koennen.

In [ ]:
out_path, _ = cleaner.cleaner(
    str(NEW_RECORDING), nut=NUT, platte=PLATTE, overwrite=True
)
df_out = pl.read_parquet(out_path)

tmp = Path(out_path).with_suffix(".bench.parquet")
write_times = []
for _ in range(REPEATS):
    t0 = time.perf_counter()
    df_out.write_parquet(tmp)
    write_times.append(time.perf_counter() - t0)
tmp.unlink(missing_ok=True)

write_times.sort()
write_s = write_times[len(write_times) // 2]
print(f"Schreibvorgang (Median): {write_s:.3f} s")
print(f"Ausgabedatei           : {dir_size_mb(out_path):.1f} MB")
del df_out
gc.collect()

175277 175273
WCS_Y_mm: 273 von 779,055 Zeilen ohne Zuordnung (Toleranz 10ms zu klein?)
Schreibvorgang (Median): 0.063 s
Ausgabedatei           : 1.5 MB


0

## 6  Normierte Gegenueberstellung

Die beiden Nuten sind unterschiedlich lang und enthalten unterschiedlich viele Kanaele,
deshalb ist die Zeit je Durchlauf allein nicht aussagekraeftig. Die normierten Groessen
beziehen den Aufwand auf die erzeugte Datenmenge.

In [10]:
df_norm = df_res.copy()
df_norm.loc[df_norm["Kette"].str.startswith("neu"), "Zeit_ohne_Schreiben_s"] = (
    df_norm["Zeit_median_s"] - write_s
)
df_norm["Zeit_ohne_Schreiben_s"] = df_norm["Zeit_ohne_Schreiben_s"].fillna(
    df_norm["Zeit_median_s"]
)

df_norm["s_je_Mio_Zeilen"] = (
    df_norm["Zeit_ohne_Schreiben_s"] / df_norm["Ausgabezeilen"] * 1e6
)
df_norm["Byte_je_Zeile"] = (
    df_norm["Speicher_median_MB"] * 1e6 / df_norm["Ausgabezeilen"]
)
df_norm["MB_je_s"] = df_norm["Eingabe_MB"] / df_norm["Zeit_ohne_Schreiben_s"]

cols = [
    "Kette", "Eingabe_MB", "Ausgabezeilen", "Zeit_median_s",
    "Zeit_ohne_Schreiben_s", "Speicher_median_MB",
    "s_je_Mio_Zeilen", "Byte_je_Zeile", "MB_je_s",
]
df_norm[cols]

,Kette,Eingabe_MB,Ausgabezeilen,Zeit_median_s,Zeit_ohne_Schreiben_s,Speicher_median_MB,s_je_Mio_Zeilen,Byte_je_Zeile,MB_je_s
0,"alt (Pandas, JSON)",6.469,813359,4.162,4.162,973.074,5.117,"1,196.365",1.554
1,"neu (DuckDB/Polars, Parquet)",3.929,779055,0.555,0.491,96.387,0.631,123.723,7.995


In [11]:
alt = df_norm.iloc[0]
neu = df_norm.iloc[1]

print("Verhaeltnis alt zu neu")
print(f"  Zeit je Mio. Zeilen : {alt['s_je_Mio_Zeilen'] / neu['s_je_Mio_Zeilen']:.1f} x")
print(f"  Speicher je Zeile   : {alt['Byte_je_Zeile'] / neu['Byte_je_Zeile']:.1f} x")
print(f"  Durchsatz (MB/s)    : {neu['MB_je_s'] / alt['MB_je_s']:.1f} x")

df_norm[cols].to_csv("messung_aufbereitung.csv", index=False)
print("\n-> messung_aufbereitung.csv geschrieben")

Verhaeltnis alt zu neu
  Zeit je Mio. Zeilen : 8.1 x
  Speicher je Zeile   : 9.7 x
  Durchsatz (MB/s)    : 5.1 x

-> messung_aufbereitung.csv geschrieben


## 7  Hochrechnung auf eine Kampagne

Beide Ketten verarbeiten je Aufzeichnung unabhaengig, der Aufwand einer Kampagne ist
daher das Vielfache eines Einzellaufs. Die Hochrechnung fuer die alte Kette ist eine
**untere Schranke**: sie enthaelt nur die Konvertierung und nichts von dem, was zwischen
den Aufzeichnungen von Hand geschah.

In [12]:
N_SLOTS = 40

for _, r in df_norm.iterrows():
    total = r["Zeit_median_s"] * N_SLOTS
    print(f"{r['Kette']:32s} {total / 60:6.1f} min fuer {N_SLOTS} Nuten")

print("\nDie Werte der alten Kette sind eine untere Schranke, siehe Text.")

alt (Pandas, JSON)                  2.8 min fuer 40 Nuten
neu (DuckDB/Polars, Parquet)        0.4 min fuer 40 Nuten

Die Werte der alten Kette sind eine untere Schranke, siehe Text.


## 8  Was diese Messung zeigt und was nicht

**Vergleichbar.** Beide Ketten loesen dieselbe Aufgabe: aus der Rohaufzeichnung einer
Nut eine fertige Tabelle erzeugen. Die normierten Groessen beziehen den Aufwand auf die
erzeugte Zeilenzahl und sind daher von der Laenge der jeweiligen Nut unabhaengig.

**Nicht vergleichbar.** Die Eingaben unterscheiden sich in Format und Umfang. Die alte
Aufzeichnung enthaelt die Metadaten in der Datei selbst, die neue haelt sie getrennt und
teilt die Stroeme auf drei Dateien auf; ein Teil des Groessenunterschieds ist damit eine
Eigenschaft des Formats und nicht der Verarbeitung. Die Kanalkonfiguration der beiden
Kampagnen ist ebenfalls verschieden.

**Nur eine Aufzeichnung je Kette.** Von der alten Kette liegt nur eine Nut in
verarbeitbarer Form vor, weshalb die Streuung ueber Aufzeichnungen hinweg unbekannt
bleibt. Die Wiederholungen messen die Streuung der Ausfuehrung, nicht die der Daten.

**Ein System, keine Nebenlast.** Alle Laeufe fanden auf demselben Rechner ohne weitere
Last statt. Der Spitzenspeicher ist ueber RSS bestimmt und enthaelt Speicher, den der
Interpreter nach einem Lauf nicht an das Betriebssystem zurueckgibt; er ist damit eher
eine obere Schaetzung.

In [ ]:
import duckdb
import polars as pl
from pathlib import Path

rec = Path("./new_data/SLD-Test_2026-07-17T13-38-45_591745/recording_2026-07-17T13-38-55_822881")

et200 = (rec / "et200.parquet").as_posix()
hf    = (rec / "hf.parquet").as_posix()

# ET200 roh
et = pl.read_parquet(et200)
print("ET200 roh:", et["timestamp"].min(), "bis", et["timestamp"].max())

# HF roh, Paketzeitstempel
con = duckdb.connect()
print("HF    roh:", con.execute(f"""
    SELECT MIN(r.ts::TIMESTAMP), MAX(r.ts::TIMESTAMP)
    FROM (SELECT unnest(records) AS r FROM read_parquet('{hf}'))
""").fetchone())
con.close()

ET200 roh: 2026-07-17 13:41:08.722458+00:00 bis 2026-07-17 13:41:17.221970+00:00


TypeError: unsupported operand type(s) for /: 'str' and 'str'